In [26]:
from nba_api.stats.endpoints import LeagueDashLineups
from nba_api.stats.endpoints import TeamPlayerOnOffDetails
from nba_api.stats.endpoints import LeagueDashPlayerStats
from nba_api.stats.static import teams

import os
import time
import random
import pandas as pd
from requests.exceptions import ConnectionError, Timeout

In [18]:
FOUR_FACTORS = [
    "EFG_PCT",
    "FTA_RATE",
    "TM_TOV_PCT",
    "OREB_PCT",
    "OPP_EFG_PCT",
    "OPP_FTA_RATE",
    "OPP_TOV_PCT",
    "OPP_OREB_PCT"
]

In [27]:
def get_lineups(season, cache_dir="cache"):

    os.makedirs(cache_dir, exist_ok=True)

    path = f"{cache_dir}/lineups_{season}.csv"

    # load cache
    if os.path.exists(path):
        print("Loading cached:", season)
        return pd.read_csv(path)


    retries = 8

    for attempt in range(retries):

        try:

            print(
                f"Downloading {season}, attempt {attempt+1}/{retries}"
            )

            df = LeagueDashLineups(
                season=season,
                season_type_all_star="Regular Season",
                measure_type_detailed_defense="Four Factors",
                per_mode_detailed="Per100Possessions",
                group_quantity=5,
                timeout=60
            ).get_data_frames()[0]


            df.to_csv(path, index=False)

            print("Saved", season)

            time.sleep(random.uniform(2,4))

            return df


        except (ConnectionError, Timeout) as e:

            wait = 10 * (attempt + 1)

            print(
                f"Connection failed. Waiting {wait}s..."
            )

            time.sleep(wait)


    raise Exception(
        f"Could not download lineup data for {season}"
    )

In [28]:
lineups = get_lineups("2021-22")

print(lineups.shape)
print(lineups.head())

Saved 2021-22
(2000, 32)
  GROUP_SET                                  GROUP_ID  \
0   Lineups     -201145-203115-203932-203999-1628420-   
1   Lineups   -101108-203109-1626164-1628969-1629028-   
2   Lineups    -201144-202711-203497-1626220-1628378-   
3   Lineups   -201143-203935-1627759-1628369-1629057-   
4   Lineups  -201976-1626156-1626157-1629020-1630162-   

                                          GROUP_NAME     TEAM_ID  \
0  J. Green - W. Barton - A. Gordon - N. Jokic - ...  1610612743   
1  C. Paul - J. Crowder - D. Booker - M. Bridges ...  1610612756   
2  M. Conley - B. Bogdanovic - R. Gobert - R. O'N...  1610612762   
3  A. Horford - M. Smart - J. Brown - J. Tatum - ...  1610612738   
4  P. Beverley - D. Russell - K. Towns - J. Vande...  1610612750   

  TEAM_ABBREVIATION  GP   W   L  W_PCT    MIN  ...  MIN_RANK  EFG_PCT_RANK  \
0               DEN  41  26  15  0.634  761.0  ...         1          5593   
1               PHX  38  31   7  0.816  754.0  ...         2       

In [45]:
def parse_player_ids(group_id):

    return [
        int(x)
        for x in group_id.split("-")
        if x
    ]

In [22]:
def weighted_average(df):

    if len(df) == 0:
        return pd.Series(
            {x: np.nan for x in FOUR_FACTORS}
        )

    return pd.Series({
        col: np.average(
            df[col],
            weights=df["MIN"]
        )
        for col in FOUR_FACTORS
    })

In [59]:
def build_player_four_factors(lineups):

    rows = []

    for team_id, team_df in lineups.groupby("TEAM_ID"):

        players = set(
            p
            for lineup in team_df["PLAYER_IDS"]
            for p in lineup
        )

        for player_id in players:

            on = team_df[
                team_df["PLAYER_IDS"]
                .apply(lambda x: player_id in x)
            ]

            off = team_df[
                team_df["PLAYER_IDS"]
                .apply(lambda x: player_id not in x)
            ]

            # skip players with no off court minutes
            if len(on) == 0 or len(off) == 0:
                continue

            on_stats = weighted_average(on)
            off_stats = weighted_average(off)

            row = {
                "PLAYER_ID": player_id,
                "TEAM_ID": team_id
            }

            for stat in FOUR_FACTORS:

                row[f"{stat}_ON"] = on_stats[stat]
                row[f"{stat}_OFF"] = off_stats[stat]

                row[f"{stat}_DIFF"] = (
                    on_stats[stat]
                    -
                    off_stats[stat]
                )
            
            row["MIN_ON"] = on["MIN"].sum()
            row["MIN_OFF"] = off["MIN"].sum()

            rows.append(row)

    return pd.DataFrame(rows)

In [60]:
seasons = [
    "2021-22",
    "2022-23",
    "2023-24",
    "2024-25",
    "2025-26"
]


all_features = []


for season in seasons:

    print("Starting", season)

    lineups = get_lineups(season)

    lineups["PLAYER_IDS"] = (
        lineups["GROUP_ID"]
        .apply(parse_player_ids)
    )

    features = build_player_four_factors(lineups)

    features["SEASON"] = season

    all_features.append(features)

    time.sleep(2)


four_factor_df = pd.concat(
    all_features,
    ignore_index=True
)

Starting 2021-22
Loading cached: 2021-22
Starting 2022-23
Loading cached: 2022-23
Starting 2023-24
Loading cached: 2023-24
Starting 2024-25
Loading cached: 2024-25
Starting 2025-26
Loading cached: 2025-26


In [ ]:
four_factor_df = four_factor_df[
    four_factor_df["MIN_ON"] + four_factor_df["MIN_OFF"] > 300
]

In [150]:
four_factor_df = four_factor_df[
    four_factor_df["MIN_ON"] > 500
]

In [36]:
from nba_api.stats.endpoints import TeamPlayerOnOffDetails


def get_on_off_target(season):

    rows = []

    for team in teams.get_teams():

        print(team["full_name"])

        data = TeamPlayerOnOffDetails(
            team_id=team["id"],
            season=season,
            measure_type_detailed_defense="Advanced",
            per_mode_detailed="Per100Possessions"
        )

        on = data.get_data_frames()[1]
        off = data.get_data_frames()[2]


        on = on[[
            "VS_PLAYER_ID",
            "VS_PLAYER_NAME",
            "NET_RATING"
        ]]

        off = off[[
            "VS_PLAYER_ID",
            "NET_RATING"
        ]]


        df = on.merge(
            off,
            on="VS_PLAYER_ID",
            suffixes=("_ON", "_OFF")
        )


        df["TEAM_ID"] = team["id"]

        df["ON_OFF_NET"] = (
            df["NET_RATING_ON"]
            -
            df["NET_RATING_OFF"]
        )

        rows.append(df)


        time.sleep(1)


    return pd.concat(rows, ignore_index=True)

In [37]:
target_rows = []

for season in seasons:

    target = get_on_off_target(season)

    target["SEASON"] = season

    target_rows.append(target)


on_off_target = pd.concat(
    target_rows,
    ignore_index=True
)

Atlanta Hawks
Boston Celtics
Cleveland Cavaliers
New Orleans Pelicans
Chicago Bulls
Dallas Mavericks
Denver Nuggets
Golden State Warriors
Houston Rockets
Los Angeles Clippers
Los Angeles Lakers
Miami Heat
Milwaukee Bucks
Minnesota Timberwolves
Brooklyn Nets
New York Knicks
Orlando Magic
Indiana Pacers
Philadelphia 76ers
Phoenix Suns
Portland Trail Blazers
Sacramento Kings
San Antonio Spurs
Oklahoma City Thunder
Toronto Raptors
Utah Jazz
Memphis Grizzlies
Washington Wizards
Detroit Pistons
Charlotte Hornets
Atlanta Hawks
Boston Celtics
Cleveland Cavaliers
New Orleans Pelicans
Chicago Bulls
Dallas Mavericks
Denver Nuggets
Golden State Warriors
Houston Rockets
Los Angeles Clippers
Los Angeles Lakers
Miami Heat
Milwaukee Bucks
Minnesota Timberwolves
Brooklyn Nets
New York Knicks
Orlando Magic
Indiana Pacers
Philadelphia 76ers
Phoenix Suns
Portland Trail Blazers
Sacramento Kings
San Antonio Spurs
Oklahoma City Thunder
Toronto Raptors
Utah Jazz
Memphis Grizzlies
Washington Wizards
Detroit Pi

In [68]:
on_off_target

,PLAYER_ID,SEASON,ON_OFF_NET
0,2207,2021-22,-103.6
1,2544,2021-22,1.1
2,2544,2022-23,8.1
3,2544,2023-24,9.0
4,2544,2024-25,-6.2
...,...,...,...
2862,1643133,2025-26,1.6
2863,1643141,2025-26,-3.9
2864,1643158,2025-26,-10.4
2865,1643253,2025-26,-13.6


In [55]:
on_off_target = on_off_target.rename(
    columns={
        "VS_PLAYER_ID": "PLAYER_ID",
        "VS_PLAYER_NAME": "PLAYER_NAME"
    }
)

on_off_target = on_off_target.groupby(
    ["PLAYER_ID","SEASON"],
    as_index=False
).agg({
    "ON_OFF_NET":"mean"
})

In [66]:
four_factor_df

,PLAYER_ID,TEAM_ID,EFG_PCT_ON,EFG_PCT_OFF,EFG_PCT_DIFF,FTA_RATE_ON,FTA_RATE_OFF,FTA_RATE_DIFF,TM_TOV_PCT_ON,TM_TOV_PCT_OFF,...,OPP_FTA_RATE_DIFF,OPP_TOV_PCT_ON,OPP_TOV_PCT_OFF,OPP_TOV_PCT_DIFF,OPP_OREB_PCT_ON,OPP_OREB_PCT_OFF,OPP_OREB_PCT_DIFF,MIN_ON,MIN_OFF,SEASON
0,201568,1610612737,0.564719,0.540972,0.023746,0.258369,0.227632,0.030737,0.099157,0.118972,...,0.018147,0.124415,0.123692,0.000722,0.276330,0.264390,0.011940,1254.0,1586.0,2021-22
1,1629027,1610612737,0.557585,0.530894,0.026691,0.247377,0.220489,0.026888,0.110551,0.109120,...,0.023674,0.123713,0.125014,-0.001301,0.264657,0.286460,-0.021803,2188.0,652.0,2021-22
2,203524,1610612737,0.585407,0.550737,0.034670,0.206627,0.241938,-0.035311,0.110610,0.110214,...,-0.008011,0.085034,0.124838,-0.039804,0.201610,0.271106,-0.069496,59.0,2781.0,2021-22
3,1628995,1610612737,0.584364,0.551071,0.033293,0.230424,0.241331,-0.010907,0.138394,0.109891,...,-0.027065,0.119727,0.124062,-0.004334,0.295000,0.269364,0.025636,33.0,2807.0,2021-22
4,1630552,1610612737,0.543533,0.551542,-0.008009,0.372667,0.239801,0.132866,0.128000,0.110033,...,-0.050882,0.114333,0.124115,-0.009781,0.520200,0.266988,0.253212,30.0,2810.0,2021-22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2723,1629012,1610612766,0.558382,0.577976,-0.019594,0.230822,0.227480,0.003342,0.162364,0.139055,...,-0.016100,0.101812,0.113643,-0.011831,0.293328,0.260714,0.032615,393.0,1701.0,2025-26
2724,1629684,1610612766,0.583464,0.572547,0.010918,0.233813,0.227017,0.006795,0.176842,0.137044,...,0.073352,0.110801,0.111541,-0.000740,0.260265,0.268090,-0.007826,336.0,1758.0,2025-26
2725,1641750,1610612766,0.562731,0.582001,-0.019270,0.220630,0.233087,-0.012457,0.155099,0.135660,...,0.007915,0.101774,0.117846,-0.016072,0.259418,0.271773,-0.012355,837.0,1257.0,2025-26
2726,1626192,1610612766,0.500000,0.574979,-0.074979,0.121000,0.229088,-0.108088,0.195000,0.142958,...,-0.077238,0.098000,0.111545,-0.013545,0.250000,0.266989,-0.016989,19.0,2075.0,2025-26


In [67]:
on_off_target.columns

Index(['PLAYER_ID', 'SEASON', 'ON_OFF_NET'], dtype='object')

In [183]:
model_df = four_factor_df.merge(
    on_off_target,
    on=[
        "PLAYER_ID",
        "SEASON"
    ],
    how="inner"
)

In [71]:
model_df.columns

Index(['PLAYER_ID', 'TEAM_ID', 'EFG_PCT_ON', 'EFG_PCT_OFF', 'EFG_PCT_DIFF',
       'FTA_RATE_ON', 'FTA_RATE_OFF', 'FTA_RATE_DIFF', 'TM_TOV_PCT_ON',
       'TM_TOV_PCT_OFF', 'TM_TOV_PCT_DIFF', 'OREB_PCT_ON', 'OREB_PCT_OFF',
       'OREB_PCT_DIFF', 'OPP_EFG_PCT_ON', 'OPP_EFG_PCT_OFF',
       'OPP_EFG_PCT_DIFF', 'OPP_FTA_RATE_ON', 'OPP_FTA_RATE_OFF',
       'OPP_FTA_RATE_DIFF', 'OPP_TOV_PCT_ON', 'OPP_TOV_PCT_OFF',
       'OPP_TOV_PCT_DIFF', 'OPP_OREB_PCT_ON', 'OPP_OREB_PCT_OFF',
       'OPP_OREB_PCT_DIFF', 'MIN_ON', 'MIN_OFF', 'SEASON', 'ON_OFF_NET'],
      dtype='object')

In [184]:
df = model_df.copy()

# keep identifiers
id_cols = [
    "PLAYER_ID",
    "PLAYER_NAME",
    "TEAM_ID",
    "SEASON"
]


# minutes played on/off
df["MIN_TOTAL"] = (
    df["MIN_ON"] +
    df["MIN_OFF"]
)


# normalize how much evidence we have
df["ON_SHARE"] = (
    df["MIN_ON"] /
    df["MIN_TOTAL"]
)


df["OFF_SHARE"] = (
    df["MIN_OFF"] /
    df["MIN_TOTAL"]
)


print(df.shape)

(1260, 33)


In [185]:
model_df = df

In [186]:
# train test split
train_df = model_df[
    model_df["SEASON"].isin(
        ["2021-22", "2022-23", "2023-24", "2024-25"]
    )
].copy()

test_df = model_df[
    model_df["SEASON"] == "2025-26"
].copy()

In [187]:
# filtering out noisy samples
train_df = train_df[
    train_df["MIN_TOTAL"] > 1000
]

test_df = test_df[
    test_df["MIN_TOTAL"] > 1000
]

In [188]:
features = [

    # offensive four factors
    # "EFG_PCT_ON",
    # "EFG_PCT_OFF",
    "EFG_PCT_DIFF",

    # "FTA_RATE_ON",
    # "FTA_RATE_OFF",
    "FTA_RATE_DIFF",

    # "TM_TOV_PCT_ON",
    # "TM_TOV_PCT_OFF",
    "TM_TOV_PCT_DIFF",

    # "OREB_PCT_ON",
    # "OREB_PCT_OFF",
    "OREB_PCT_DIFF",


    # defensive four factors
    # "OPP_EFG_PCT_ON",
    # "OPP_EFG_PCT_OFF",
    "OPP_EFG_PCT_DIFF",

    # "OPP_FTA_RATE_ON",
    # "OPP_FTA_RATE_OFF",
    "OPP_FTA_RATE_DIFF",

    # "OPP_TOV_PCT_ON",
    # "OPP_TOV_PCT_OFF",
    "OPP_TOV_PCT_DIFF",

    # "OPP_OREB_PCT_ON",
    # "OPP_OREB_PCT_OFF",
    "OPP_OREB_PCT_DIFF",


    # sample size controls
    # "MIN_ON",
    # "MIN_OFF",
    "MIN_TOTAL",
    "ON_SHARE"

]

In [189]:
X_train = train_df[features]
y_train = train_df["ON_OFF_NET"]

X_test = test_df[features]
y_test = test_df["ON_OFF_NET"]

In [190]:
season_order = {
    "2021-22":0,
    "2022-23":1,
    "2023-24":2,
    "2024-25":3,
    "2025-26":4
}

model_df["SEASON_NUM"] = (
    model_df["SEASON"]
    .map(season_order)
)

model_df = model_df.sort_values("SEASON_NUM")



In [177]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
import numpy as np


tscv = TimeSeriesSplit(
    n_splits=3
)


alphas = [
    .01,
    .1,
    1,
    10,
    50,
    100
]


for alpha in alphas:

    model = Pipeline([
        (
            "scale",
            StandardScaler()
        ),
        (
            "ridge",
            Ridge(alpha=alpha)
        )
    ])


    scores = []


    for train_idx, val_idx in tscv.split(X):

        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]

        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]


        model.fit(
            X_train,
            y_train
        )


        pred = model.predict(
            X_val
        )


        mae = mean_absolute_error(
            y_val,
            pred
        )

        scores.append(mae)


    print(
        alpha,
        np.mean(scores)
    )

0.01 4.683569648896017
0.1 4.683514946470377
1 4.68296968722937
10 4.6786041043277296
50 4.668314611037925
100 4.664042287409659


In [191]:
model = Pipeline([
        (
            "scale",
            StandardScaler()
        ),
        (
            "ridge",
            Ridge(alpha=100)
        )
    ])

In [192]:
model.fit(X_train,y_train)

,steps,"[('scale', ...), ('ridge', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,alpha,100
,fit_intercept,True
,copy_X,True
,max_iter,None


In [193]:
predictions = model.predict(X_test)
results_df = test_df.assign(Predicted_Value=predictions)


In [195]:
from nba_api.stats.endpoints import LeagueDashPlayerStats

seasons = [
    "2021-22",
    "2022-23",
    "2023-24",
    "2024-25",
    "2025-26"
]

name_map = {}

for season in seasons:
    df = LeagueDashPlayerStats(
        season=season,
        per_mode_detailed="PerGame"
    ).get_data_frames()[0]

    season_map = (
        df[["PLAYER_ID", "PLAYER_NAME"]]
        .drop_duplicates()
        .set_index("PLAYER_ID")["PLAYER_NAME"]
        .to_dict()
    )

    name_map.update(season_map)

model_df["PLAYER_NAME"] = model_df["PLAYER_ID"].map(name_map)


In [196]:
results_df["PLAYER_NAME"] = results_df["PLAYER_ID"].map(name_map)

In [166]:
results_df.columns

Index(['PLAYER_ID', 'TEAM_ID', 'EFG_PCT_ON', 'EFG_PCT_OFF', 'EFG_PCT_DIFF',
       'FTA_RATE_ON', 'FTA_RATE_OFF', 'FTA_RATE_DIFF', 'TM_TOV_PCT_ON',
       'TM_TOV_PCT_OFF', 'TM_TOV_PCT_DIFF', 'OREB_PCT_ON', 'OREB_PCT_OFF',
       'OREB_PCT_DIFF', 'OPP_EFG_PCT_ON', 'OPP_EFG_PCT_OFF',
       'OPP_EFG_PCT_DIFF', 'OPP_FTA_RATE_ON', 'OPP_FTA_RATE_OFF',
       'OPP_FTA_RATE_DIFF', 'OPP_TOV_PCT_ON', 'OPP_TOV_PCT_OFF',
       'OPP_TOV_PCT_DIFF', 'OPP_OREB_PCT_ON', 'OPP_OREB_PCT_OFF',
       'OPP_OREB_PCT_DIFF', 'MIN_ON', 'MIN_OFF', 'SEASON', 'ON_OFF_NET',
       'MIN_TOTAL', 'ON_SHARE', 'OFF_SHARE', 'Predicted_Value', 'PLAYER_NAME'],
      dtype='object')

In [197]:
missing = results_df[results_df["PLAYER_NAME"].isna()]
print(missing[["PLAYER_ID"]])

Empty DataFrame
Columns: [PLAYER_ID]
Index: []


In [203]:
mae = mean_absolute_error(
            y_test,
            predictions
        )

In [204]:
mae

2.8524530790356346

In [198]:
results_df = results_df[['PLAYER_NAME', 'Predicted_Value']]
df_sorted = results_df.sort_values(by='Predicted_Value', ascending=False)
df_sorted

,PLAYER_NAME,Predicted_Value
1193,Victor Wembanyama,12.504722
1023,CJ McCollum,11.643199
1022,Nickeil Alexander-Walker,11.469749
1121,Giannis Antetokounmpo,11.339690
1209,Chet Holmgren,11.141002
...,...,...
1040,De'Andre Hunter,-7.146679
1251,Ronald Holland II,-7.945810
1249,Isaiah Stewart,-8.264888
1021,Vít Krejčí,-9.442261


In [199]:
df_sorted.to_csv("outputs/four_factor_model_results_v2.csv", index=False)

In [201]:
regression_pipe_coefs = pd.DataFrame(
    data=model.named_steps["ridge"].coef_, index=features, columns=["Coefficient"]
)

In [202]:
regression_pipe_coefs.sort_values(by="Coefficient", ascending=False).head(80)

,Coefficient
EFG_PCT_DIFF,2.319666
OREB_PCT_DIFF,1.192281
OPP_TOV_PCT_DIFF,0.977830
ON_SHARE,0.966644
FTA_RATE_DIFF,0.726970
MIN_TOTAL,0.024032
OPP_FTA_RATE_DIFF,-0.840632
TM_TOV_PCT_DIFF,-0.970172
OPP_OREB_PCT_DIFF,-1.045074
OPP_EFG_PCT_DIFF,-2.010175
